Step 1: Connect to Databases

In [35]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection details
username = "root"
password = "nour"
host = "localhost"
port = 3306 #default MySQL port
database = "sakila"
dw_db = "movie_dw"

# Create connection string
connection_string = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"

# Connect to Sakila OLTP
engine_sakila = create_engine(connection_string)

# Connect to Data Warehouse
engine_dw = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{dw_db}")


Step 2: Build Dimension Tables

In [36]:
# --- Dimensions ---
# Customer Dimension
customer_df = pd.read_sql("""
SELECT c.customer_id, c.first_name, c.last_name, c.email,
       ci.city, co.country
FROM customer c
JOIN address a ON c.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
""", engine_sakila)
customer_df['full_name'] = customer_df['first_name'] + " " + customer_df['last_name']

# Film Dimension
film_df = pd.read_sql("""
SELECT f.film_id, f.title, f.release_year, f.length, f.rating,
       l.name AS language
FROM film f
JOIN language l ON f.language_id = l.language_id
""", engine_sakila)

# Store Dimension
store_df = pd.read_sql("""
SELECT s.store_id, ci.city, co.country
FROM store s
JOIN address a ON s.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
""", engine_sakila)

# Staff Dimension
staff_df = pd.read_sql("SELECT staff_id, first_name, last_name, email, store_id FROM staff", engine_sakila)

# Date Dimension (all unique dates from rentals, returns, payments)
dates_df = pd.read_sql("""
SELECT rental_date, return_date FROM rental
UNION
SELECT payment_date, NULL FROM payment
""", engine_sakila)

all_dates = pd.to_datetime(dates_df.stack(), errors='coerce').dropna().unique()
date_df = pd.DataFrame(all_dates, columns=['date']).drop_duplicates().sort_values('date').reset_index(drop=True)
date_df['day'] = date_df['date'].dt.day
date_df['month'] = date_df['date'].dt.month
date_df['quarter'] = date_df['date'].dt.quarter
date_df['year'] = date_df['date'].dt.year
date_df['weekday'] = date_df['date'].dt.day_name()

Step 3: Build Fact Tables

In [37]:
# --- Facts ---
# Rental Fact
rental_df = pd.read_sql("SELECT * FROM rental", engine_sakila)
rental_df['rental_duration'] = (
    pd.to_datetime(rental_df['return_date']) - pd.to_datetime(rental_df['rental_date'])
).dt.days
rental_df['late_return_flag'] = rental_df['rental_duration'] > 3
fact_rental = rental_df[['rental_id','customer_id','inventory_id','staff_id',
                         'rental_date','return_date','rental_duration','late_return_flag']]

# Payment Fact
payment_df = pd.read_sql("SELECT * FROM payment", engine_sakila)
fact_payment = payment_df[['payment_id','customer_id','staff_id','rental_id',
                           'amount','payment_date']]

# Inventory Fact
inventory_df = pd.read_sql("SELECT inventory_id, film_id, store_id FROM inventory", engine_sakila)
fact_inventory = inventory_df[['inventory_id','film_id','store_id']]

Step 4: Load into Warehouse

In [38]:
# --- Load ---
# Dimensions
customer_df.to_sql("dim_customer", engine_dw, if_exists="replace", index=False)
film_df.to_sql("dim_film", engine_dw, if_exists="replace", index=False)
store_df.to_sql("dim_store", engine_dw, if_exists="replace", index=False)
staff_df.to_sql("dim_staff", engine_dw, if_exists="replace", index=False)
date_df.to_sql("dim_date", engine_dw, if_exists="replace", index=False)

# Facts
fact_rental.to_sql("fact_rental", engine_dw, if_exists="replace", index=False)
fact_payment.to_sql("fact_payment", engine_dw, if_exists="replace", index=False)
fact_inventory.to_sql("fact_inventory", engine_dw, if_exists="replace", index=False)

print("Dimensional model ETL completed successfully!")

Dimensional model ETL completed successfully!


In [47]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection details
username = "root"
password = "nour"
host = "localhost"
port = 3306
database = "sakila"
dw_db = "movie_dw"

# Connect to Sakila
engine_sakila = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
# Connect to Data Warehouse
engine_dw = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{dw_db}")

# --- Dimensions ---
# Customer Dimension
customer_df = pd.read_sql("""
SELECT c.customer_id, c.first_name, c.last_name, c.email,
       ci.city, co.country
FROM customer c
JOIN address a ON c.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
""", engine_sakila)
customer_df['full_name'] = customer_df['first_name'] + " " + customer_df['last_name']

# Film Dimension
film_df = pd.read_sql("""
SELECT f.film_id, f.title, f.release_year, f.length, f.rating,
       l.name AS language
FROM film f
JOIN language l ON f.language_id = l.language_id
""", engine_sakila)

# Store Dimension
store_df = pd.read_sql("""
SELECT s.store_id, ci.city, co.country
FROM store s
JOIN address a ON s.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
""", engine_sakila)

# Staff Dimension
staff_df = pd.read_sql("SELECT staff_id, first_name, last_name, email, store_id FROM staff", engine_sakila)

# Date Dimension (optional, for time analysis)
dates_df = pd.read_sql("""
SELECT rental_date, return_date FROM rental
UNION
SELECT payment_date, NULL FROM payment
""", engine_sakila)

all_dates = pd.to_datetime(dates_df.stack(), errors='coerce').dropna().unique()
date_df = pd.DataFrame(all_dates, columns=['date']).drop_duplicates().sort_values('date').reset_index(drop=True)
date_df['day'] = date_df['date'].dt.day
date_df['month'] = date_df['date'].dt.month
date_df['quarter'] = date_df['date'].dt.quarter
date_df['year'] = date_df['date'].dt.year
date_df['weekday'] = date_df['date'].dt.day_name()

# --- Facts ---
# Rental Fact
rental_df = pd.read_sql("SELECT * FROM rental", engine_sakila)
rental_df['rental_duration'] = (
    pd.to_datetime(rental_df['return_date']) - pd.to_datetime(rental_df['rental_date'])
).dt.days
rental_df['late_return_flag'] = rental_df['rental_duration'] > 3
fact_rental = rental_df[['rental_id','customer_id','inventory_id','staff_id',
                         'rental_date','return_date','rental_duration','late_return_flag']]

# Payment Fact
payment_df = pd.read_sql("SELECT * FROM payment", engine_sakila)
fact_payment = payment_df[['payment_id','customer_id','staff_id','rental_id',
                           'amount','payment_date']]

# Inventory Fact
inventory_df = pd.read_sql("SELECT inventory_id, film_id, store_id FROM inventory", engine_sakila)
fact_inventory = inventory_df[['inventory_id','film_id','store_id']]

# --- Load ---
customer_df.to_sql("dim_customer", engine_dw, if_exists="replace", index=False)
film_df.to_sql("dim_film", engine_dw, if_exists="replace", index=False)
store_df.to_sql("dim_store", engine_dw, if_exists="replace", index=False)
staff_df.to_sql("dim_staff", engine_dw, if_exists="replace", index=False)
date_df.to_sql("dim_date", engine_dw, if_exists="replace", index=False)

fact_rental.to_sql("fact_rental", engine_dw, if_exists="replace", index=False)
fact_payment.to_sql("fact_payment", engine_dw, if_exists="replace", index=False)
fact_inventory.to_sql("fact_inventory", engine_dw, if_exists="replace", index=False)

print("ETL completed successfully — clean version without foreign keys!")


ETL completed successfully — clean version without foreign keys!
